In [1]:

import numpy as np
import pandas as pd

# Reproducible random data
np.random.seed(42)

n = 1000

# Generate customer features
tenure = np.random.randint(1, 73, n)

monthly_charges = np.random.randint(500, 5001, n)

support_tickets = np.random.poisson(2, n)

late_payments = np.random.poisson(1, n)

# Create a probability of customer churn
logit = (
    -1.5
    - 0.035 * tenure
    + 0.00025 * monthly_charges
    + 0.45 * support_tickets
    + 0.55 * late_payments
)

churn_probability = 1 / (1 + np.exp(-logit))

# Generate binary target
churn = np.random.binomial(
    1,
    churn_probability
)

# Create DataFrame
df = pd.DataFrame({
    "tenure": tenure,
    "monthly_charges": monthly_charges,
    "support_tickets": support_tickets,
    "late_payments": late_payments,
    "churn": churn
})

print(df.head())
print("\nDataset shape:", df.shape)
print("\nChurn distribution:")
print(df["churn"].value_counts())


   tenure  monthly_charges  support_tickets  late_payments  churn
0      52             3962                3              2      0
1      15             3579                0              1      1
2      72             1332                3              1      0
3      61             4908                2              0      0
4      21             3306                3              0      0

Dataset shape: (1000, 5)

Churn distribution:
churn
0    625
1    375
Name: count, dtype: int64


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   tenure           1000 non-null   int32
 1   monthly_charges  1000 non-null   int32
 2   support_tickets  1000 non-null   int32
 3   late_payments    1000 non-null   int32
 4   churn            1000 non-null   int32
dtypes: int32(5)
memory usage: 19.7 KB


In [3]:
df.describe()

,tenure,monthly_charges,support_tickets,late_payments,churn
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,35.881000,2770.028000,2.007000,1.010000,0.375000
std,20.958915,1327.458541,1.399611,0.990394,0.484365
min,1.000000,501.000000,0.000000,0.000000,0.000000
25%,17.750000,1546.750000,1.000000,0.000000,0.000000
50%,35.500000,2807.500000,2.000000,1.000000,0.000000
75%,54.000000,3952.250000,3.000000,2.000000,1.000000
max,72.000000,4997.000000,9.000000,5.000000,1.000000


In [4]:
df.isnull().sum()

tenure             0
monthly_charges    0
support_tickets    0
late_payments      0
churn              0
dtype: int64

In [5]:
df["churn"].value_counts(normalize=True)

churn
0    0.625
1    0.375
Name: proportion, dtype: float64

In [6]:
X = df.drop("churn", axis=1)

y = df["churn"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   tenure  monthly_charges  support_tickets  late_payments
0      52             3962                3              2
1      15             3579                0              1
2      72             1332                3              1
3      61             4908                2              0
4      21             3306                3              0

Target:
0    0
1    1
2    0
3    0
4    0
Name: churn, dtype: int32


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 800
Testing records: 200


In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

# Train the model
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [9]:
predictions = model.predict(X_test)

print(predictions[:20])

[0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0]


In [10]:
probabilities = model.predict_proba(X_test)

print(probabilities[:5])

[[0.64789023 0.35210977]
 [0.91868903 0.08131097]
 [0.38311864 0.61688136]
 [0.76077492 0.23922508]
 [0.6132894  0.3867106 ]]


In [11]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.71

Confusion Matrix:
[[106  19]
 [ 39  36]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.85      0.79       125
           1       0.65      0.48      0.55        75

    accuracy                           0.71       200
   macro avg       0.69      0.66      0.67       200
weighted avg       0.70      0.71      0.70       200

